<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/model_evaluation/numerical_answer_evaluation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Numerical Answer Evaluation using LLM-as-Judge

This notebook demonstrates how to evaluate LLM-generated answers that contain numerical values by:
1. **Extracting numbers** from both model answers and ground truth
2. **Matching corresponding number pairs** based on context
3. **Computing statistical metrics** to assess numerical accuracy
4. **Deciding acceptance/rejection** based on configurable thresholds

## Use Cases:
- Evaluating math problem solutions
- Fact-checking numerical claims in generated text
- Comparing financial/scientific data in LLM outputs
- Quality control for RAG systems with numerical data

## Metrics Used:
- **Relative Error**: `|predicted - actual| / |actual|`
- **Absolute Error**: `|predicted - actual|`
- **Percentage Accuracy**: Based on acceptable tolerance
- **Order of Magnitude Check**: Ensures numbers are in the right ballpark


## 1. Installation and Setup


In [ ]:
# Install required packages
!pip install -q pandas numpy matplotlib seaborn plotly scipy


In [ ]:
import re
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Statistics
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✅ All imports successful!")


## 2. Number Extraction Module

This module extracts numbers from text, handling various formats:
- Integers: `42`, `1000`, `1,000`
- Decimals: `3.14`, `0.5`, `.25`
- Percentages: `50%`, `12.5%`
- Scientific notation: `1.5e6`, `3.2E-4`
- Currency: `$100`, `€50.00`
- Fractions: `1/2`, `3/4`


In [ ]:
@dataclass
class ExtractedNumber:
    """Represents a number extracted from text with its context."""
    value: float
    original_text: str
    start_pos: int
    end_pos: int
    context_before: str = ""
    context_after: str = ""
    number_type: str = "unknown"  # integer, decimal, percentage, scientific, currency, fraction
    
    def __repr__(self):
        return f"ExtractedNumber({self.value}, '{self.original_text}', type={self.number_type})"


class NumberExtractor:
    """
    Extracts numbers from text with context information.
    """
    
    def __init__(self, context_window: int = 30):
        """
        Args:
            context_window: Number of characters to capture before/after each number
        """
        self.context_window = context_window
        
        # Regex patterns for different number formats
        self.patterns = {
            # Scientific notation (must come first to avoid partial matches)
            'scientific': r'[-+]?\d+\.?\d*[eE][-+]?\d+',
            # Percentage
            'percentage': r'[-+]?\d+\.?\d*\s*%',
            # Currency (common symbols)
            'currency': r'[$€£¥₹]\s*\d{1,3}(?:,\d{3})*(?:\.\d+)?|\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*(?:dollars?|euros?|pounds?)',
            # Fraction
            'fraction': r'\d+\s*/\s*\d+',
            # Decimal with commas (e.g., 1,234.56)
            'decimal_comma': r'[-+]?\d{1,3}(?:,\d{3})*\.\d+',
            # Integer with commas (e.g., 1,234)
            'integer_comma': r'[-+]?\d{1,3}(?:,\d{3})+(?!\.\d)',
            # Regular decimal
            'decimal': r'[-+]?\d+\.\d+',
            # Regular integer
            'integer': r'[-+]?\d+',
        }
        
        # Compile patterns
        self.compiled_patterns = {
            name: re.compile(pattern) 
            for name, pattern in self.patterns.items()
        }
    
    def _parse_number(self, text: str, number_type: str) -> float:
        """Convert extracted text to float value."""
        text = text.strip()
        
        if number_type == 'percentage':
            # Remove % and convert
            return float(text.replace('%', '').replace(',', '').strip())
        
        elif number_type == 'currency':
            # Remove currency symbols and words
            cleaned = re.sub(r'[$€£¥₹]|\b(?:dollars?|euros?|pounds?)\b', '', text)
            return float(cleaned.replace(',', '').strip())
        
        elif number_type == 'fraction':
            # Parse fraction
            parts = text.split('/')
            return float(parts[0]) / float(parts[1])
        
        elif number_type in ['decimal_comma', 'integer_comma']:
            # Remove commas
            return float(text.replace(',', ''))
        
        elif number_type == 'scientific':
            return float(text)
        
        else:
            return float(text.replace(',', ''))
    
    def extract(self, text: str) -> List[ExtractedNumber]:
        """
        Extract all numbers from text.
        
        Returns:
            List of ExtractedNumber objects with context
        """
        extracted = []
        used_positions = set()  # Track positions to avoid overlapping matches
        
        # Process patterns in order of specificity
        for number_type in ['scientific', 'percentage', 'currency', 'fraction', 
                           'decimal_comma', 'integer_comma', 'decimal', 'integer']:
            pattern = self.compiled_patterns[number_type]
            
            for match in pattern.finditer(text):
                start, end = match.start(), match.end()
                
                # Skip if overlaps with already extracted number
                if any(start < used_end and end > used_start 
                       for used_start, used_end in used_positions):
                    continue
                
                original_text = match.group()
                
                try:
                    value = self._parse_number(original_text, number_type)
                    
                    # Extract context
                    context_start = max(0, start - self.context_window)
                    context_end = min(len(text), end + self.context_window)
                    
                    context_before = text[context_start:start].strip()
                    context_after = text[end:context_end].strip()
                    
                    extracted.append(ExtractedNumber(
                        value=value,
                        original_text=original_text,
                        start_pos=start,
                        end_pos=end,
                        context_before=context_before,
                        context_after=context_after,
                        number_type=number_type
                    ))
                    
                    used_positions.add((start, end))
                    
                except (ValueError, ZeroDivisionError):
                    continue
        
        # Sort by position in text
        extracted.sort(key=lambda x: x.start_pos)
        return extracted

# Test the extractor
extractor = NumberExtractor()

test_text = """
The company reported revenue of $2.5 billion in Q3 2023, representing a 15% increase 
from Q2. The profit margin improved to 23.4%, up from 21.8% in the previous quarter.
Total employees: 45,000. Stock price: $142.50. Market cap: 1.2e11 dollars.
The ratio of debt to equity is 3/4.
"""

numbers = extractor.extract(test_text)

print("📊 Number Extraction Demo")
print("="*70)
print(f"\nInput text:\n{test_text}")
print(f"\nExtracted {len(numbers)} numbers:")
for num in numbers:
    print(f"  • {num.value:,.4g} ({num.number_type}): '{num.original_text}'")
    print(f"    Context: '...{num.context_before}' [{num.original_text}] '{num.context_after}...'")


## 3. Number Matching and Pairing

This module matches numbers from the model answer to corresponding numbers in the ground truth based on:
1. **Position-based matching**: Numbers in similar relative positions
2. **Context-based matching**: Numbers with similar surrounding text
3. **Type-based matching**: Same number type (percentage to percentage, etc.)


In [ ]:
@dataclass
class NumberPair:
    """A matched pair of numbers from model answer and ground truth."""
    model_number: ExtractedNumber
    truth_number: ExtractedNumber
    match_score: float  # Confidence in the match (0-1)
    match_reason: str


class NumberMatcher:
    """
    Matches numbers between model answer and ground truth.
    """
    
    def __init__(self):
        # Common words that indicate what a number represents
        self.indicator_words = {
            'revenue': ['revenue', 'sales', 'income'],
            'profit': ['profit', 'earnings', 'margin', 'net income'],
            'percentage': ['percent', '%', 'rate', 'growth', 'increase', 'decrease'],
            'count': ['total', 'number', 'count', 'employees', 'users', 'customers'],
            'price': ['price', 'cost', 'value', '$', 'dollar', 'euro'],
            'time': ['year', 'month', 'quarter', 'q1', 'q2', 'q3', 'q4', 'date'],
            'ratio': ['ratio', 'proportion', 'fraction'],
        }
    
    def _get_context_keywords(self, num: ExtractedNumber) -> set:
        """Extract keywords from number context."""
        context = (num.context_before + " " + num.context_after).lower()
        words = set(re.findall(r'\b\w+\b', context))
        return words
    
    def _calculate_context_similarity(self, num1: ExtractedNumber, num2: ExtractedNumber) -> float:
        """Calculate similarity between contexts of two numbers."""
        words1 = self._get_context_keywords(num1)
        words2 = self._get_context_keywords(num2)
        
        if not words1 or not words2:
            return 0.0
        
        # Jaccard similarity
        intersection = len(words1 & words2)
        union = len(words1 | words2)
        
        return intersection / union if union > 0 else 0.0
    
    def _get_semantic_category(self, num: ExtractedNumber) -> Optional[str]:
        """Determine semantic category of a number based on context."""
        context = (num.context_before + " " + num.context_after).lower()
        
        for category, keywords in self.indicator_words.items():
            if any(kw in context for kw in keywords):
                return category
        
        # Also check number type
        if num.number_type == 'percentage':
            return 'percentage'
        elif num.number_type == 'currency':
            return 'price'
        elif num.number_type == 'fraction':
            return 'ratio'
        
        return None
    
    def match(self, model_numbers: List[ExtractedNumber], 
              truth_numbers: List[ExtractedNumber],
              strategy: str = "hybrid") -> List[NumberPair]:
        """
        Match numbers between model answer and ground truth.
        
        Args:
            model_numbers: Numbers extracted from model answer
            truth_numbers: Numbers extracted from ground truth
            strategy: "position", "context", or "hybrid"
            
        Returns:
            List of matched NumberPair objects
        """
        if strategy == "position":
            return self._match_by_position(model_numbers, truth_numbers)
        elif strategy == "context":
            return self._match_by_context(model_numbers, truth_numbers)
        else:
            return self._match_hybrid(model_numbers, truth_numbers)
    
    def _match_by_position(self, model_numbers: List[ExtractedNumber], 
                          truth_numbers: List[ExtractedNumber]) -> List[NumberPair]:
        """Match numbers based on their position in text."""
        pairs = []
        n = min(len(model_numbers), len(truth_numbers))
        
        for i in range(n):
            pairs.append(NumberPair(
                model_number=model_numbers[i],
                truth_number=truth_numbers[i],
                match_score=0.5,  # Medium confidence for position-based
                match_reason="position"
            ))
        
        return pairs
    
    def _match_by_context(self, model_numbers: List[ExtractedNumber], 
                         truth_numbers: List[ExtractedNumber]) -> List[NumberPair]:
        """Match numbers based on context similarity."""
        pairs = []
        used_truth = set()
        
        for model_num in model_numbers:
            best_match = None
            best_score = 0.0
            
            model_category = self._get_semantic_category(model_num)
            
            for i, truth_num in enumerate(truth_numbers):
                if i in used_truth:
                    continue
                
                # Check type compatibility
                type_bonus = 0.2 if model_num.number_type == truth_num.number_type else 0.0
                
                # Check semantic category
                truth_category = self._get_semantic_category(truth_num)
                category_bonus = 0.3 if model_category and model_category == truth_category else 0.0
                
                # Context similarity
                context_sim = self._calculate_context_similarity(model_num, truth_num)
                
                total_score = context_sim * 0.5 + type_bonus + category_bonus
                
                if total_score > best_score:
                    best_score = total_score
                    best_match = (i, truth_num)
            
            if best_match and best_score > 0.2:  # Minimum threshold
                used_truth.add(best_match[0])
                pairs.append(NumberPair(
                    model_number=model_num,
                    truth_number=best_match[1],
                    match_score=best_score,
                    match_reason="context"
                ))
        
        return pairs
    
    def _match_hybrid(self, model_numbers: List[ExtractedNumber], 
                     truth_numbers: List[ExtractedNumber]) -> List[NumberPair]:
        """Combine position and context-based matching."""
        # First try context-based matching
        context_pairs = self._match_by_context(model_numbers, truth_numbers)
        
        # For unmatched numbers, fall back to position
        matched_model = {p.model_number.start_pos for p in context_pairs}
        matched_truth = {p.truth_number.start_pos for p in context_pairs}
        
        unmatched_model = [n for n in model_numbers if n.start_pos not in matched_model]
        unmatched_truth = [n for n in truth_numbers if n.start_pos not in matched_truth]
        
        position_pairs = self._match_by_position(unmatched_model, unmatched_truth)
        
        return context_pairs + position_pairs

# Test the matcher
matcher = NumberMatcher()

ground_truth = """
The company reported quarterly revenue of $2.5 billion, with a profit margin of 23.5%.
Total workforce: 45,000 employees. Year-over-year growth was 15%.
"""

model_answer = """
Revenue for the quarter was $2.4 billion, and the profit margin reached 24.1%.
The company employs approximately 44,500 people. Growth rate: 14.8% compared to last year.
"""

truth_nums = extractor.extract(ground_truth)
model_nums = extractor.extract(model_answer)

print("📊 Number Matching Demo")
print("="*70)
print(f"\nGround Truth: {ground_truth.strip()}")
print(f"\nModel Answer: {model_answer.strip()}")

pairs = matcher.match(model_nums, truth_nums)

print(f"\n\nMatched {len(pairs)} number pairs:")
for pair in pairs:
    print(f"\n  Model: {pair.model_number.value:,.4g} ('{pair.model_number.original_text}')")
    print(f"  Truth: {pair.truth_number.value:,.4g} ('{pair.truth_number.original_text}')")
    print(f"  Match: {pair.match_reason} (confidence: {pair.match_score:.2f})")


## 4. Statistical Comparison Module

This module computes various statistical metrics to compare number pairs and determine if the model answer is acceptable.


In [ ]:
class AcceptanceDecision(Enum):
    """Decision on whether a numerical answer is acceptable."""
    ACCEPT = "accept"
    MARGINAL = "marginal"
    REJECT = "reject"


@dataclass
class PairComparison:
    """Statistical comparison of a number pair."""
    pair: NumberPair
    absolute_error: float
    relative_error: float
    percentage_error: float
    order_of_magnitude_diff: float
    is_within_tolerance: bool
    decision: AcceptanceDecision
    details: Dict[str, Any] = field(default_factory=dict)


@dataclass
class EvaluationResult:
    """Overall evaluation result for a model answer."""
    question: str
    model_answer: str
    ground_truth: str
    pair_comparisons: List[PairComparison]
    overall_score: float
    overall_decision: AcceptanceDecision
    summary: Dict[str, Any]


class NumericalEvaluator:
    """
    Evaluates numerical accuracy of model answers against ground truth.
    """
    
    def __init__(self, 
                 relative_tolerance: float = 0.10,  # 10% relative error
                 absolute_tolerance: float = 0.01,  # Small numbers tolerance
                 order_magnitude_tolerance: float = 1.0,  # Same order of magnitude
                 acceptance_threshold: float = 0.7,  # 70% of numbers must be accurate
                 marginal_threshold: float = 0.5):
        """
        Args:
            relative_tolerance: Maximum acceptable relative error (0.1 = 10%)
            absolute_tolerance: Maximum acceptable absolute error for small numbers
            order_magnitude_tolerance: Maximum acceptable order of magnitude difference
            acceptance_threshold: Fraction of accurate numbers for ACCEPT
            marginal_threshold: Fraction of accurate numbers for MARGINAL
        """
        self.relative_tolerance = relative_tolerance
        self.absolute_tolerance = absolute_tolerance
        self.order_magnitude_tolerance = order_magnitude_tolerance
        self.acceptance_threshold = acceptance_threshold
        self.marginal_threshold = marginal_threshold
        
        self.extractor = NumberExtractor()
        self.matcher = NumberMatcher()
    
    def _calculate_metrics(self, model_value: float, truth_value: float) -> Dict[str, float]:
        """Calculate all comparison metrics for a number pair."""
        # Absolute error
        abs_error = abs(model_value - truth_value)
        
        # Relative error (handle division by zero)
        if truth_value == 0:
            rel_error = float('inf') if model_value != 0 else 0.0
        else:
            rel_error = abs_error / abs(truth_value)
        
        # Percentage error
        pct_error = rel_error * 100
        
        # Order of magnitude difference
        if model_value == 0 or truth_value == 0:
            if model_value == truth_value:
                magnitude_diff = 0.0
            else:
                magnitude_diff = float('inf')
        else:
            magnitude_diff = abs(np.log10(abs(model_value)) - np.log10(abs(truth_value)))
        
        return {
            'absolute_error': abs_error,
            'relative_error': rel_error,
            'percentage_error': pct_error,
            'magnitude_diff': magnitude_diff
        }
    
    def _is_within_tolerance(self, metrics: Dict[str, float], truth_value: float) -> bool:
        """Check if the comparison is within acceptable tolerance."""
        # For small numbers, use absolute tolerance
        if abs(truth_value) < 1.0:
            return metrics['absolute_error'] <= self.absolute_tolerance
        
        # For large numbers, use relative tolerance
        return (metrics['relative_error'] <= self.relative_tolerance and
                metrics['magnitude_diff'] <= self.order_magnitude_tolerance)
    
    def _make_pair_decision(self, metrics: Dict[str, float], 
                           truth_value: float) -> AcceptanceDecision:
        """Make acceptance decision for a single pair."""
        if self._is_within_tolerance(metrics, truth_value):
            return AcceptanceDecision.ACCEPT
        
        # Marginal if within 2x tolerance
        if metrics['relative_error'] <= self.relative_tolerance * 2:
            return AcceptanceDecision.MARGINAL
        
        return AcceptanceDecision.REJECT
    
    def compare_pair(self, pair: NumberPair) -> PairComparison:
        """Compare a single number pair."""
        model_val = pair.model_number.value
        truth_val = pair.truth_number.value
        
        metrics = self._calculate_metrics(model_val, truth_val)
        is_within = self._is_within_tolerance(metrics, truth_val)
        decision = self._make_pair_decision(metrics, truth_val)
        
        return PairComparison(
            pair=pair,
            absolute_error=metrics['absolute_error'],
            relative_error=metrics['relative_error'],
            percentage_error=metrics['percentage_error'],
            order_of_magnitude_diff=metrics['magnitude_diff'],
            is_within_tolerance=is_within,
            decision=decision,
            details={
                'model_value': model_val,
                'truth_value': truth_val,
                'tolerance_used': 'absolute' if abs(truth_val) < 1.0 else 'relative'
            }
        )
    
    def evaluate(self, question: str, model_answer: str, 
                 ground_truth: str) -> EvaluationResult:
        """
        Evaluate a model answer against ground truth.
        
        Returns:
            EvaluationResult with detailed comparison
        """
        # Extract numbers
        model_nums = self.extractor.extract(model_answer)
        truth_nums = self.extractor.extract(ground_truth)
        
        # Match numbers
        pairs = self.matcher.match(model_nums, truth_nums)
        
        # Compare each pair
        comparisons = [self.compare_pair(pair) for pair in pairs]
        
        # Calculate overall metrics
        if comparisons:
            accurate_count = sum(1 for c in comparisons if c.is_within_tolerance)
            accuracy_ratio = accurate_count / len(comparisons)
            
            avg_relative_error = np.mean([c.relative_error for c in comparisons 
                                          if c.relative_error != float('inf')])
            avg_abs_error = np.mean([c.absolute_error for c in comparisons])
        else:
            accuracy_ratio = 1.0 if not truth_nums else 0.0
            avg_relative_error = 0.0
            avg_abs_error = 0.0
        
        # Overall decision
        if accuracy_ratio >= self.acceptance_threshold:
            overall_decision = AcceptanceDecision.ACCEPT
        elif accuracy_ratio >= self.marginal_threshold:
            overall_decision = AcceptanceDecision.MARGINAL
        else:
            overall_decision = AcceptanceDecision.REJECT
        
        # Calculate overall score (0-1)
        if comparisons:
            # Weighted by match confidence
            weighted_scores = []
            for c in comparisons:
                if c.decision == AcceptanceDecision.ACCEPT:
                    score = 1.0
                elif c.decision == AcceptanceDecision.MARGINAL:
                    score = 0.5
                else:
                    score = 0.0
                weighted_scores.append(score * c.pair.match_score)
            
            total_weight = sum(c.pair.match_score for c in comparisons)
            overall_score = sum(weighted_scores) / total_weight if total_weight > 0 else 0.0
        else:
            overall_score = 1.0 if not truth_nums else 0.0
        
        summary = {
            'total_numbers_in_truth': len(truth_nums),
            'total_numbers_in_model': len(model_nums),
            'matched_pairs': len(pairs),
            'accurate_pairs': sum(1 for c in comparisons if c.is_within_tolerance),
            'accuracy_ratio': accuracy_ratio,
            'avg_relative_error': avg_relative_error,
            'avg_absolute_error': avg_abs_error,
            'decisions': {
                'accept': sum(1 for c in comparisons if c.decision == AcceptanceDecision.ACCEPT),
                'marginal': sum(1 for c in comparisons if c.decision == AcceptanceDecision.MARGINAL),
                'reject': sum(1 for c in comparisons if c.decision == AcceptanceDecision.REJECT)
            }
        }
        
        return EvaluationResult(
            question=question,
            model_answer=model_answer,
            ground_truth=ground_truth,
            pair_comparisons=comparisons,
            overall_score=overall_score,
            overall_decision=overall_decision,
            summary=summary
        )

# Test the evaluator
evaluator = NumericalEvaluator(relative_tolerance=0.10)  # 10% tolerance

question = "What were the company's Q3 2023 financial results?"

ground_truth = """
The company reported quarterly revenue of $2.5 billion, with a profit margin of 23.5%.
Total workforce: 45,000 employees. Year-over-year growth was 15%.
"""

model_answer = """
Revenue for the quarter was $2.4 billion, and the profit margin reached 24.1%.
The company employs approximately 44,500 people. Growth rate: 14.8% compared to last year.
"""

result = evaluator.evaluate(question, model_answer, ground_truth)

print("📊 Numerical Evaluation Result")
print("="*70)
print(f"\nQuestion: {question}")
print(f"\nOverall Decision: {result.overall_decision.value.upper()}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"\nSummary:")
for key, value in result.summary.items():
    print(f"  • {key}: {value}")


In [ ]:
# Detailed comparison view
print("\n📋 Detailed Pair Comparisons:")
print("="*70)

for i, comp in enumerate(result.pair_comparisons, 1):
    status_icon = "✅" if comp.decision == AcceptanceDecision.ACCEPT else (
        "⚠️" if comp.decision == AcceptanceDecision.MARGINAL else "❌"
    )
    
    print(f"\n{status_icon} Pair {i}:")
    print(f"   Model:  {comp.details['model_value']:,.4g} ('{comp.pair.model_number.original_text}')")
    print(f"   Truth:  {comp.details['truth_value']:,.4g} ('{comp.pair.truth_number.original_text}')")
    print(f"   Absolute Error: {comp.absolute_error:,.4g}")
    print(f"   Relative Error: {comp.relative_error:.2%}")
    print(f"   Decision: {comp.decision.value.upper()}")


## 5. Sample Evaluation Dataset

Let's create a diverse dataset of questions with ground truth and model answers to test the evaluator.


In [ ]:
# Create diverse evaluation examples
evaluation_examples = [
    {
        "id": 1,
        "question": "What are the key statistics for the 2023 Olympic Games?",
        "ground_truth": "The 2023 Olympics featured 10,500 athletes from 206 countries competing in 329 events. The host country invested $15.4 billion in infrastructure. Medal count: USA 113, China 89, UK 64.",
        "good_answer": "The Olympics had about 10,400 athletes from 205 nations participating in 330 events. Infrastructure investment was approximately $15.5 billion. Top medal counts were USA with 112, China with 90, and UK with 63 medals.",
        "poor_answer": "Around 8,000 athletes from 150 countries competed in 200 events. The investment was $5 billion. USA won 80 medals, China 50, UK 30."
    },
    {
        "id": 2,
        "question": "What is the current state of global renewable energy?",
        "ground_truth": "Global renewable energy capacity reached 3,372 GW in 2023, a 9.6% increase from 2022. Solar accounts for 1,419 GW (42.1%), wind for 899 GW (26.7%). Investment totaled $495 billion.",
        "good_answer": "Renewable capacity hit 3,400 GW globally in 2023, up 10% year-over-year. Solar leads at 1,420 GW (42%), followed by wind at 900 GW (27%). Total investment was around $500 billion.",
        "poor_answer": "Global renewables are at 5,000 GW capacity with 20% growth. Solar is 2,500 GW and wind is 1,500 GW. Investment exceeded $1 trillion."
    },
    {
        "id": 3,
        "question": "Calculate the area of a rectangle with length 12.5m and width 8.4m.",
        "ground_truth": "The area of the rectangle is 105 square meters (12.5 × 8.4 = 105).",
        "good_answer": "The rectangular area equals 105 sq meters, calculated as 12.5m times 8.4m.",
        "poor_answer": "The area is approximately 150 square meters (12.5 × 8.4 ≈ 150)."
    },
    {
        "id": 4,
        "question": "What were Apple's Q4 2023 earnings?",
        "ground_truth": "Apple reported Q4 2023 revenue of $89.5 billion, up 1.5% YoY. iPhone revenue: $43.8 billion. Services: $22.3 billion. Gross margin: 45.2%. EPS: $1.46.",
        "good_answer": "Apple's Q4 revenue was $89.6 billion, a 1.4% increase. iPhone brought in $43.9 billion, Services $22.2 billion. Gross margin stood at 45.1%, with EPS of $1.45.",
        "poor_answer": "Apple made $120 billion in Q4, with iPhone at $60 billion. Services contributed $10 billion. Margin was 35% and EPS was $2.50."
    },
    {
        "id": 5,
        "question": "What is the distance from Earth to Mars?",
        "ground_truth": "The distance from Earth to Mars varies from 54.6 million km (closest) to 401 million km (farthest). Average distance is about 225 million km. Light takes 12.5 minutes on average.",
        "good_answer": "Earth-Mars distance ranges from 55 million km at closest approach to 400 million km at farthest. The average is roughly 225 million km, with light taking about 12-13 minutes to travel between them.",
        "poor_answer": "Mars is about 50 million km from Earth at all times. Light takes 5 minutes to reach Mars from Earth."
    }
]

# Convert to DataFrame
df_examples = pd.DataFrame(evaluation_examples)

print(f"📊 Created {len(df_examples)} evaluation examples")
print("\nExamples:")
for _, row in df_examples.iterrows():
    print(f"\n  {row['id']}. {row['question'][:60]}...")


## 6. Comprehensive Evaluation


In [ ]:
# Evaluate all examples
print("🔄 Running comprehensive evaluation...")
print("="*70)

all_results = []

for _, row in df_examples.iterrows():
    # Evaluate good answer
    good_result = evaluator.evaluate(row['question'], row['good_answer'], row['ground_truth'])
    all_results.append({
        'example_id': row['id'],
        'question': row['question'][:50] + '...',
        'answer_type': 'Good',
        'overall_score': good_result.overall_score,
        'decision': good_result.overall_decision.value,
        'accuracy_ratio': good_result.summary['accuracy_ratio'],
        'avg_rel_error': good_result.summary['avg_relative_error'],
        'matched_pairs': good_result.summary['matched_pairs']
    })
    
    # Evaluate poor answer
    poor_result = evaluator.evaluate(row['question'], row['poor_answer'], row['ground_truth'])
    all_results.append({
        'example_id': row['id'],
        'question': row['question'][:50] + '...',
        'answer_type': 'Poor',
        'overall_score': poor_result.overall_score,
        'decision': poor_result.overall_decision.value,
        'accuracy_ratio': poor_result.summary['accuracy_ratio'],
        'avg_rel_error': poor_result.summary['avg_relative_error'],
        'matched_pairs': poor_result.summary['matched_pairs']
    })
    
    print(f"  ✓ Example {row['id']}: Good={good_result.overall_decision.value}, Poor={poor_result.overall_decision.value}")

results_df = pd.DataFrame(all_results)

print("\n✅ Evaluation complete!")
print(f"\n📊 Results Summary:")
print(results_df.to_string(index=False))


## 7. Visualizations


In [ ]:
# Create visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Overall Score by Answer Type",
        "Decision Distribution",
        "Accuracy Ratio Comparison",
        "Average Relative Error"
    ),
    specs=[
        [{"type": "bar"}, {"type": "pie"}],
        [{"type": "bar"}, {"type": "bar"}]
    ]
)

colors = {"Good": "#2ecc71", "Poor": "#e74c3c"}

# Plot 1: Overall Score by Answer Type
for answer_type in ['Good', 'Poor']:
    data = results_df[results_df['answer_type'] == answer_type]
    fig.add_trace(
        go.Bar(
            name=answer_type,
            x=[f"Ex {i}" for i in data['example_id']],
            y=data['overall_score'],
            marker_color=colors[answer_type]
        ),
        row=1, col=1
    )

# Plot 2: Decision Distribution
decision_counts = results_df['decision'].value_counts()
fig.add_trace(
    go.Pie(
        labels=decision_counts.index,
        values=decision_counts.values,
        marker_colors=['#2ecc71', '#f39c12', '#e74c3c'],
        hole=0.4
    ),
    row=1, col=2
)

# Plot 3: Accuracy Ratio Comparison
for answer_type in ['Good', 'Poor']:
    data = results_df[results_df['answer_type'] == answer_type]
    fig.add_trace(
        go.Bar(
            name=f"{answer_type} Accuracy",
            x=[f"Ex {i}" for i in data['example_id']],
            y=data['accuracy_ratio'],
            marker_color=colors[answer_type],
            showlegend=False
        ),
        row=2, col=1
    )

# Plot 4: Average Relative Error
for answer_type in ['Good', 'Poor']:
    data = results_df[results_df['answer_type'] == answer_type]
    fig.add_trace(
        go.Bar(
            name=f"{answer_type} Error",
            x=[f"Ex {i}" for i in data['example_id']],
            y=data['avg_rel_error'],
            marker_color=colors[answer_type],
            showlegend=False
        ),
        row=2, col=2
    )

fig.update_layout(
    title_text="Numerical Answer Evaluation Results",
    showlegend=True,
    height=700,
    width=1100,
    barmode='group'
)

fig.update_yaxes(title_text="Score (0-1)", row=1, col=1)
fig.update_yaxes(title_text="Ratio", row=2, col=1)
fig.update_yaxes(title_text="Relative Error", row=2, col=2)

fig.show()


## 8. Configurable Tolerance Levels

Different use cases require different tolerance levels. Let's explore how changing tolerances affects decisions.


In [ ]:
# Test different tolerance configurations
tolerance_configs = {
    "Strict (5%)": NumericalEvaluator(relative_tolerance=0.05),
    "Standard (10%)": NumericalEvaluator(relative_tolerance=0.10),
    "Lenient (20%)": NumericalEvaluator(relative_tolerance=0.20),
    "Very Lenient (30%)": NumericalEvaluator(relative_tolerance=0.30)
}

# Use example 1 for testing
test_example = df_examples.iloc[0]

print("📊 Tolerance Sensitivity Analysis")
print("="*70)
print(f"\nQuestion: {test_example['question']}")
print(f"\nGround Truth: {test_example['ground_truth'][:100]}...")
print(f"\nGood Answer: {test_example['good_answer'][:100]}...")
print(f"\nPoor Answer: {test_example['poor_answer'][:100]}...")

print("\n" + "="*70)
print(f"{'Configuration':<20} {'Good Answer':<15} {'Poor Answer':<15} {'Good Score':<12} {'Poor Score':<12}")
print("="*70)

for config_name, eval_instance in tolerance_configs.items():
    good_result = eval_instance.evaluate(
        test_example['question'], 
        test_example['good_answer'], 
        test_example['ground_truth']
    )
    poor_result = eval_instance.evaluate(
        test_example['question'], 
        test_example['poor_answer'], 
        test_example['ground_truth']
    )
    
    print(f"{config_name:<20} {good_result.overall_decision.value:<15} {poor_result.overall_decision.value:<15} {good_result.overall_score:<12.2f} {poor_result.overall_score:<12.2f}")


## 9. Best Practices and Recommendations

### Tolerance Selection Guidelines:

| Use Case | Recommended Tolerance | Rationale |
|----------|----------------------|-----------|
| Financial reporting | 1-2% | Exact numbers matter |
| Scientific calculations | 5% | Precision important |
| General fact-checking | 10% | Reasonable approximations |
| Casual comparisons | 20% | Order of magnitude matters |

### Key Considerations:

1. **Number Type Matters**: Percentages vs. absolute numbers may need different tolerances
2. **Context is Important**: Match numbers based on semantic meaning, not just position
3. **Handle Edge Cases**: Zero values, very large/small numbers need special handling
4. **Multiple Numbers**: Consider overall accuracy ratio, not just individual pairs

### When to Use This Evaluator:

✅ **Good for:**
- Fact-checking numerical claims
- Evaluating math problem solutions
- Comparing financial/scientific data
- Quality control for data-heavy RAG systems

⚠️ **Limitations:**
- Requires ground truth with exact numbers
- May not capture nuanced numerical relationships
- Context matching can be imperfect for complex texts


In [ ]:
# Final summary
print("✅ Numerical Answer Evaluation Demo Complete!")
print("="*70)
print("\n📋 Components Implemented:")
print("   1. NumberExtractor     - Extracts numbers from text with context")
print("   2. NumberMatcher       - Matches number pairs between texts")
print("   3. NumericalEvaluator  - Compares pairs with statistical metrics")
print("   4. AcceptanceDecision  - Accept/Marginal/Reject classification")

print("\n📊 Metrics Available:")
print("   • Absolute Error")
print("   • Relative Error")
print("   • Percentage Error")
print("   • Order of Magnitude Difference")
print("   • Accuracy Ratio")
print("   • Overall Score")

print("\n⚙️ Configurable Parameters:")
print("   • relative_tolerance (default: 10%)")
print("   • absolute_tolerance (default: 0.01)")
print("   • order_magnitude_tolerance (default: 1.0)")
print("   • acceptance_threshold (default: 70%)")
print("   • marginal_threshold (default: 50%)")

print("\n🔗 Resources:")
print("   - Statistical comparison methods")
print("   - Number extraction with regex")
print("   - Context-based number matching")
